# 02. Prompt Engineering 실습
**SK하이닉스 Autonomous R&D — Day 3** 

---
## 0. 준비

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))


def ask(system_prompt, user_prompt, temperature=0.2):
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=temperature,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ],
    )
    return response.choices[0].message.content

In [2]:
# 기본적으로 같은 모델이라도 프롬프트에 따라 다른 출력을 생성

system = 'You are a helpful assistant. Answer in Korean.'

prompts_ko = [
    "프랑스의 수도는",
    "Q: 프랑스의 수도는 어디인가요? A:",
    "프랑스의 수도 도시는",
]

for prompt in prompts_ko:
    print(f"프롬프트: {prompt}")
    print(f"생성: {ask(system, prompt, temperature=0.2)}")
    print()

프롬프트: 프랑스의 수도는
생성: 프랑스의 수도는 파리입니다.

프롬프트: Q: 프랑스의 수도는 어디인가요? A:
생성: 프랑스의 수도는 파리입니다.

프롬프트: 프랑스의 수도 도시는
생성: 프랑스의 수도 도시는 파리입니다.



---
## 1. Zero-shot vs Few-shot 

| 방식 | 설명 |
|------|------|
| **Zero-shot** | 예시 없이 지시만 |
| **Few-shot** | 원하는 형식의 **예시**를 함께 제공 |

In [3]:
system = 'You are a helpful assistant. Answer in Korean.'
# Zero-shot — 형식 지시 없음
user_zero = '''
아래 영화 리뷰가 긍정인지 부정인지 판정해줘.
- "연출은 좋았지만 2시간이 너무 길었다"
- "최고의 영화, 또 보고 싶다"
'''
print('=== Zero-shot ===')
print(ask(system, user_zero))

=== Zero-shot ===
첫 번째 리뷰는 부정적인 요소가 포함되어 있어 부정으로 판정할 수 있습니다. 두 번째 리뷰는 긍정적인 감정을 표현하고 있어 긍정으로 판정할 수 있습니다.


In [4]:
# Few-shot — 출력 형식 예시 포함
user_few = '''
아래 형식으로 감성 판정해줘.

[예시]
입력: "배우 연기가 훌륭했다" → 긍정 | 9/10
입력: "스토리가 지루했다" → 부정 | 3/10

[실제 데이터]
- "연출은 좋았지만 2시간이 너무 길었다"
- "최고의 영화, 또 보고 싶다"
'''
print('=== Few-shot ===')
print(ask(system, user_few))

=== Few-shot ===
입력: "연출은 좋았지만 2시간이 너무 길었다" → 긍정 | 6/10  
입력: "최고의 영화, 또 보고 싶다" → 긍정 | 10/10


In [5]:
zero_shot_prompt_ko = """다음은 단어 유추의 예시입니다:

문제: 행복한 : 슬픈 :: 좋은 :"""

print(ask(system, zero_shot_prompt_ko, temperature=0.7))

좋은 : 나쁜


In [6]:
few_shot_prompt_ko = """다음은 단어 유추의 예시입니다:

뜨거운 : 차가운 :: 위 : 아래
큰 : 작은 :: 빠른 : 느린
낮 : 밤 :: 밝은 : 어두운
행복한 : 슬픈 :: 좋은 :"""

print('Few-shot 프롬프트:')
print(few_shot_prompt_ko)
print('\n생성된 완성:')
print(ask(system, few_shot_prompt_ko, temperature=0.7))

Few-shot 프롬프트:
다음은 단어 유추의 예시입니다:

뜨거운 : 차가운 :: 위 : 아래
큰 : 작은 :: 빠른 : 느린
낮 : 밤 :: 밝은 : 어두운
행복한 : 슬픈 :: 좋은 :

생성된 완성:
나쁜


---
## 2. 역할(Role) 설정 

In [7]:
question = '하루 커피 4잔 마셔도 괜찮을까?'

print('=== 일반 assistant ===')
print(ask('You are a helpful assistant.', question))
print('\n' + '=' * 50 + '\n')

=== 일반 assistant ===
하루에 커피 4잔을 마시는 것은 일반적으로 건강한 성인에게는 괜찮다고 여겨집니다. 많은 연구에 따르면, 하루 400mg 이하의 카페인 섭취는 대부분의 사람들에게 안전하다고 합니다. 일반적인 커피 한 잔에는 약 95mg의 카페인이 들어 있으므로, 하루 4잔은 약 380mg의 카페인에 해당합니다.

하지만 개인의 카페인 민감도는 다를 수 있으며, 일부 사람들은 카페인에 더 민감하게 반응할 수 있습니다. 또한, 임신 중이거나 특정 건강 문제가 있는 경우에는 카페인 섭취를 제한하는 것이 좋습니다. 

따라서, 본인의 몸 상태와 반응을 고려하여 적절한 양을 조절하는 것이 중요합니다. 커피를 마신 후 불안감, 불면증, 심장 두근거림 등의 증상이 나타난다면 섭취량을 줄이는 것이 좋습니다.




In [8]:
print('=== 영양사 역할 ===')
print(ask(
    'You are a registered dietitian. Answer in Korean with caffeine mg estimate and health advice.',
    question,
))

=== 영양사 역할 ===
하루에 커피 4잔을 마시는 것은 일반적으로 건강한 성인에게는 괜찮다고 여겨집니다. 보통 한 잔의 커피에는 약 95mg의 카페인이 들어있으므로, 4잔을 마신다면 약 380mg의 카페인을 섭취하게 됩니다. 

미국 식품의약국(FDA)에서는 건강한 성인이 하루에 최대 400mg의 카페인을 섭취하는 것이 안전하다고 권장하고 있습니다. 하지만 개인의 카페인 민감도, 건강 상태, 임신 여부 등에 따라 다를 수 있으므로, 자신의 몸 상태를 잘 살펴보는 것이 중요합니다.

카페인은 일시적으로 에너지를 높여줄 수 있지만, 과도한 섭취는 불안감, 불면증, 심장 두근거림 등의 부작용을 초래할 수 있습니다. 따라서 커피를 마실 때는 적당한 양을 유지하고, 수분 섭취도 충분히 하며, 카페인 섭취 후의 몸 상태를 잘 관찰하는 것이 좋습니다.


In [9]:
roles = [
    "당신은 친절한 고객 서비스 상담원입니다.",
    "당신은 전문적인 기술 문서 작성자입니다.",
    "당신은 창의적인 소설가입니다."
]

question = "인공지능에 대해 설명해주세요."

for role in roles:
    user_prompt = f"{role}\n\n질문: {question}\n\n답변:"
    print(f"=== {role} ===")
    print(f"답변: {ask('Answer in Korean.', user_prompt, temperature=0.8)}")
    print()

=== 당신은 친절한 고객 서비스 상담원입니다. ===
답변: 인공지능(AI)은 컴퓨터 시스템이 인간처럼 지능적인 작업을 수행할 수 있도록 하는 기술입니다. 이는 기계 학습, 자연어 처리, 이미지 인식 등 다양한 분야를 포함하며, 데이터를 분석하고 패턴을 인식하여 결정을 내리거나 문제를 해결하는 능력을 갖추고 있습니다.

예를 들어, 인공지능은 음성 인식 기술을 통해 사용자의 말을 이해하고 반응할 수 있으며, 추천 시스템을 통해 사용자 맞춤형 정보를 제공하기도 합니다. 또한, 자율주행차와 같은 첨단 기술에도 활용되고 있어, 미래의 다양한 산업에서 큰 영향을 미칠 것으로 기대됩니다.

인공지능은 이미 우리의 일상생활에 널리 퍼져 있으며, 앞으로도 더욱 발전하여 다양한 분야에서 혁신적인 변화를 가져올 것입니다. 추가로 궁금한 점이 있으시면 언제든지 질문해 주세요!

=== 당신은 전문적인 기술 문서 작성자입니다. ===
답변: 인공지능(AI, Artificial Intelligence)은 컴퓨터 시스템이나 기계가 인간의 지능을 모방하여 학습, 추론, 문제 해결, 이해, 언어 처리 등의 기능을 수행할 수 있도록 하는 기술입니다. AI는 데이터와 알고리즘을 기반으로 하여 다양한 작업을 자동화하고 효율성을 높이는 데 기여합니다.

AI는 주로 크게 두 가지 유형으로 나눌 수 있습니다:

1. **좁은 인공지능(Narrow AI)**: 특정 작업에 특화된 인공지능으로, 예를 들어 음성 인식, 이미지 인식, 추천 시스템 등이 있습니다. 이러한 시스템은 특정 문제를 해결하는 데 뛰어난 성능을 보이지만, 다른 분야로의 일반화는 어렵습니다.

2. **일반 인공지능(General AI)**: 인간과 유사한 수준의 지능을 갖춘 AI로, 다양한 작업을 수행할 수 있는 능력을 지닌 것을 의미합니다. 현재까지 연구와 개발이 진행 중이며, 실제로 구현된 사례는 없습니다.

AI의 주요 기술에는 머신 러닝(Machine Learning), 딥 러닝(Deep Learning), 자연어 처리

---
## 3. 출력 형식 지정 

In [10]:
topic = '재택근무의 장단점 3가지'
system_ko = 'Answer in Korean.'

print('=== Markdown ===')
print(ask(system_ko, topic + '\n형식: markdown bullet point'))
print('\n' + '=' * 50 + '\n')

=== Markdown ===
### 재택근무의 장단점

#### 장점
- **유연한 근무 시간**: 개인의 일정에 맞춰 근무 시간을 조정할 수 있어 일과 삶의 균형을 맞추기 용이함.
- **교통 시간 절약**: 출퇴근 시간이 없으므로 시간을 절약하고 스트레스를 줄일 수 있음.
- **편안한 근무 환경**: 자신이 선호하는 환경에서 일할 수 있어 집중력과 생산성을 높일 수 있음.

#### 단점
- **사회적 고립감**: 동료와의 직접적인 소통이 줄어들어 외로움을 느낄 수 있음.
- **업무와 개인 생활의 경계 모호**: 집에서 일하다 보면 업무와 개인 생활의 경계가 흐려져 집중하기 어려울 수 있음.
- **자기 관리 필요**: 스스로 동기부여를 해야 하며, 시간 관리와 업무 관리를 잘 하지 않으면 생산성이 떨어질 수 있음.




In [11]:
print('=== Table ===')
print(ask(
    system_ko,
    topic + '\n형식: markdown 표 (항목 | 장점 | 단점)만 출력',
))

=== Table ===
| 항목       | 장점                           | 단점                           |
|------------|--------------------------------|--------------------------------|
| 유연성     | 근무 시간을 자유롭게 조정 가능 | 일과 삶의 경계가 모호해질 수 있음 |
| 통근 시간  | 통근 시간 절약으로 생산성 증가 | 집에서의 방해 요소가 많을 수 있음 |
| 비용 절감  | 교통비 및 식비 절감            | 가정에서의 전기세 등 추가 비용 발생 가능 |


---
### 출력 형식 (표 / JSON)

In [12]:
import json

topic = '식각(Etch) 공정에서 수율에 영향 주는 요인 3가지'
system_ko = 'Answer in Korean.'

print('=== Table ===')
print(ask(
    system_ko,
    topic + '\n형식: markdown 표 (요인 | 설명 | 대응)만 출력',
))

=== Table ===
| 요인         | 설명                                   | 대응                       |
|--------------|----------------------------------------|----------------------------|
| 화학물질     | 식각에 사용되는 화학물질의 농도와 순도 | 화학물질 품질 관리 및 최적화 |
| 공정 조건    | 온도, 압력, 시간 등 공정 조건의 변화 | 공정 조건 모니터링 및 조정  |
| 마스크 품질  | 마스크의 정밀도와 결함 여부          | 마스크 검사 및 품질 보증   |


In [13]:
print('=== JSON ===')
result = ask(
    'Output ONLY valid JSON. No markdown.',
    topic + '\n{"factors": [{"name": "", "action": ""}]} 형식으로',
)
print(result)
try:
    print('\n파싱 성공:', list(json.loads(result).keys()))
except json.JSONDecodeError:
    print('\n파싱 실패 — ONLY valid JSON 강조 필요')

=== JSON ===
{
  "factors": [
    {
      "name": "화학물질의 농도",
      "action": "적절한 농도를 유지하여 식각 속도와 균일성을 확보"
    },
    {
      "name": "식각 시간",
      "action": "최적의 식각 시간을 설정하여 과식각이나 부족식각 방지"
    },
    {
      "name": "온도 및 압력",
      "action": "일정한 온도와 압력을 유지하여 공정의 일관성 확보"
    }
  ]
}

파싱 성공: ['factors']


---
## 4. Chain-of-Thought (CoT)

CoT는 모델이 단계별로 추론 과정을 보여주도록 하는 기법.

In [15]:
## zero-shot 예시

problem = '''
A팀 10명, B팀 15명, C팀 5명.
전체 25명 중 40% 이상이 A팀이면 A팀에 추가 인원 배치.
지금 추가 배치가 필요한가?
'''
system = 'You are a helpful assistant. Answer in Korean.'
print('=== CoT 없음 ===')
print(ask(system, problem))

=== CoT 없음 ===
전체 25명 중 40%는 10명입니다. 현재 A팀은 10명으로, 전체 인원의 40%에 해당합니다. 따라서 A팀의 인원이 40% 이상이 아니므로 추가 배치가 필요하지 않습니다.


In [16]:
print('=== CoT 적용 ===')
print(ask(
    system + ' Show step-by-step reasoning before final answer.',
    problem + '\n\n단계별로 생각해 봅시다.',
))

=== CoT 적용 ===
먼저, 전체 인원 수와 A팀의 인원 수를 확인해 보겠습니다.

1. **전체 인원 수**: A팀 10명 + B팀 15명 + C팀 5명 = 30명
2. **A팀 인원 수**: 10명

다음으로, A팀의 인원이 전체 인원에서 차지하는 비율을 계산해 보겠습니다.

3. **A팀 비율 계산**:
   - A팀 비율 = (A팀 인원 수 / 전체 인원 수) × 100
   - A팀 비율 = (10 / 30) × 100 = 33.33%

이제 A팀의 비율이 40% 이상인지 확인해 보겠습니다.

4. **비율 비교**:
   - A팀 비율 33.33%는 40% 미만입니다.

따라서, A팀의 인원이 전체 인원 중 40% 이상이 아니므로 추가 배치가 필요하지 않습니다.

결론적으로, **A팀에 추가 배치가 필요하지 않습니다.**


In [17]:
problem = """다음 처럼 문제를 답하시오.

문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

답: ?원"""

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

답: 3,000원


In [18]:
problem = """다음 처럼 문제를 답하시오.

문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

천천히 단계별로 생각해봅시다.

답: ?원"""

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

먼저, 연필과 지우개의 가격을 계산해보겠습니다.

1. 연필의 가격:
   - 연필 1자루의 가격은 500원입니다.
   - 연필 5자루의 가격은 500원 × 5자루 = 2500원입니다.

2. 지우개의 가격:
   - 지우개 1개의 가격은 300원입니다.
   - 지우개 3개의 가격은 300원 × 3개 = 900원입니다.

3. 총 금액 계산:
   - 총 금액은 연필의 가격과 지우개의 가격을 더한 것입니다.
   - 총 금액 = 2500원 + 900원 = 3400원입니다.

따라서, 총 금액은 3400원입니다.

답: 3400원


In [19]:
problem = """다음 처럼 문제를 답하시오.

문제: 한 상점에서 사과 3개와 배 2개를 샀습니다. 사과는 개당 1000원, 배는 개당 1500원입니다. 총 금액은 얼마인가요?

답: 6000원

---

문제: 한 상점에서 귤 1개와 바나나 12개를 샀습니다. 사과는 개당 1000원, 배는 개당 200원입니다. 총 금액은 얼마인가요?

답: 3400원

---


문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

답: """

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

답: 3900원


In [20]:
## one-shot 예시

problem = """다음 문제를 단계별로 풀어보세요.

문제: 한 상점에서 사과 3개와 배 2개를 샀습니다. 사과는 개당 1000원, 배는 개당 1500원입니다. 총 금액은 얼마인가요?

단계별 풀이:
1. 사과 3개의 가격: 3 × 1000 = 3000원
2. 배 2개의 가격: 2 × 1500 = 3000원
3. 총 금액: 3000 + 3000 = 6000원
답: 6000원

---

문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

단계별 풀이:"""

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

단계별 풀이:

1. 연필 5자루의 가격: 5 × 500 = 2500원
2. 지우개 3개의 가격: 3 × 300 = 900원
3. 총 금액: 2500 + 900 = 3400원

답: 3400원


---
## 5. System + User 프롬프트 

In [21]:
system_prompt = '''
너는 스타트업 PM의 일정 관리 AI다.
규칙: 결론 먼저, bullet 3개, 한국어
'''

user_prompt = '''
이번 주 마감: 월-기획안, 수-코드리뷰, 금-발표.
목요일에 하루 종일 회의가 잡혔어. 우선순위 조정해줘.
'''

print(ask(system_prompt, user_prompt))

결론: 목요일 회의에 맞춰 우선순위를 조정하자.

- 월요일: 기획안 작성 완료
- 수요일: 코드리뷰 진행
- 금요일: 발표 준비 및 리허설


In [22]:
system_md = (
    "너는 백화점 MD 업무를 보조하는 AI다. "
    "답변은 반드시 표 형태로 작성하고, "
    "결론을 먼저 제시한 뒤 "
    "리스크와 가정을 명시해야 한다."
)

user_md = "이번 주 VIP 이탈 상위 200명에 대한 액션 플랜을 작성해줘."

answer = ask(system_md, user_md, temperature=0.7)
print(answer)

| 결론                                       |
|------------------------------------------|
| VIP 이탈 상위 200명에 대한 맞춤형 리텐션 전략을 마련하여 이탈 방지 및 재유치에 집중한다. |

| 리스크                                      | 가정                                      |
|------------------------------------------|------------------------------------------|
| 1. VIP 고객의 불만 사항이 반영되지 않을 경우 이탈이 지속될 수 있다. | 1. VIP 고객은 개인화된 서비스를 중시하며, 이에 대한 반응이 긍정적일 것이다. |
| 2. 리텐션 전략 실행에 필요한 자원이 부족할 수 있다. | 2. 고객의 이탈 원인이 가격, 서비스 품질, 제품 다양성 등으로 명확하게 파악될 것이다. |
| 3. 특정 VIP 고객층에 대한 분석이 불충분할 수 있다. | 3. VIP 고객의 재구매 의도가 높은 경우가 많을 것이다. |

| 액션 플랜                                      |
|------------------------------------------|
| 1. 고객 세분화: 이탈 원인을 분석하고, 고객의 세분화 진행. |
| 2. 개인화된 커뮤니케이션: 맞춤형 이메일 및 문자 발송. |
| 3. 특별 프로모션: VIP 고객 전용 할인 및 이벤트 제공. |
| 4. 피드백 수집: 고객 만족도 조사 및 불만 사항 개선. |
| 5. 재방문 유도: VIP 고객을 위한 특별한 런칭 이벤트 초대. |
| 6. 고객 관리 강화: 전담 CS팀 운영 및 지속적인 연락 유지. |


---
## 6. Self-check Prompting

In [ ]:
check_prompt = f"""아래는 네가 작성한 'VIP 이탈 상위 200명 액션 플랜' 초안이다.

[초안]
{answer}

이제 아래 체크리스트로 점검하고, 필요하면 수정본을 작성하라.

체크리스트:
1) 논리적 모순/비약이 있는가?
2) 실행 불가능하거나 모호한 액션이 있는가? (담당/기한/방법이 불명확한지)
3) 과도한 가정이 있는가?
4) 표 형식이 지켜졌는가? (결론 먼저, 리스크/가정 포함)

출력 규칙:
- 먼저 "점검 결과"를 bullet로 간단히 쓰고
- 그 다음 "수정본"을 작성하라
- 수정본은 반드시 표 형태 + 결론 먼저 + 리스크/가정 명시
"""

final_answer = ask(system_md, check_prompt, temperature=0.7)
print('\n--- Self-check 후 최종 답변(수정본) ---')
print(final_answer)

---
## 7. Constraint Prompting

In [ ]:
system_prompt = (
    "너는 백화점 MD 업무를 보조하는 AI다. "
    "답변은 반드시 표 형태로 작성하고, "
    "결론을 먼저 제시한 뒤, "
    "리스크와 가정을 명시해야 한다."
)

constraints = """조건(반드시 준수):
- 추가 예산은 최대 1억 원 이내
- 인력 증원 불가 (기존 인력 내에서 운영)
- 2주 이내 실행 가능한 액션만 제시
- 고객에게 직접 연락하는 액션은 과도한 빈도를 피하고(1회 이내),
  개인정보/컴플라이언스 리스크를 고려할 것
"""

user_prompt = f"""이번 주 VIP 이탈 상위 200명에 대한 액션 플랜을 작성해줘.

{constraints}

출력 형식:
- 결론(한 줄) 먼저
- 그 다음 표로 정리 (컬럼 예: 타깃/액션/채널/우선순위/기간/기대효과/담당)
- 마지막에 리스크/가정 명시
"""

constrained_answer = ask(system_prompt, user_prompt, temperature=0.7)
print('\n--- 조건 기반 최종 답변(답변만) ---')
print(constrained_answer)